# Retraining the bird species classifier

The original classifier was a small CNN trained on **64x64** crops, which is a lot of
detail to throw away - in the project report it calls an Indian Peacock at 47% and a
Cattle Egret at 23%.

This notebook fine-tunes a pretrained backbone at **224x224** instead, on the same 25
species, and exports both a PyTorch checkpoint and an ONNX file (the web app runs the
ONNX one in the browser).

It runs on **Kaggle** or **Colab** - the next cell works out which and points everything
at the right folders.

**On Kaggle** - the easier one: the dataset is already there and the run doesn't need you
watching it.

1. *Add Input* -> search `birds25-cleaned` -> add pavangawande's dataset.
2. Settings -> *Accelerator*: **GPU**, and *Internet*: **on**. Internet is needed for the
   pretrained ImageNet weights and for `pip install onnxruntime`; turning it on needs a
   phone-verified account.
3. *Save Version* -> **Save & Run All (Commit)**. It runs on Kaggle's machines with the
   browser closed, and the files land in that version's Output tab.

**On Colab**: Runtime -> Change runtime type -> **GPU**, then run the cells. It asks for
your `kaggle.json` and downloads the dataset (~16 GB, a few minutes).

**If it drops you halfway.** Colab will, especially with the tab in the background. The
training cell writes a checkpoint after every epoch - to Drive on Colab, to the working
folder on Kaggle - and picks up from the last one when you rerun it, so a disconnect costs
one epoch rather than the whole run.

Dataset: [birds25-cleaned](https://www.kaggle.com/datasets/pavangawande/birds25-cleaned)
by pavangawande, CC BY-NC 4.0 - non-commercial use only, which is what this is.

## 0. Where is this running

Kaggle mounts the dataset read-only under `/kaggle/input` and keeps whatever is in
`/kaggle/working`; Colab has an empty disk and Drive. Everything below reads the paths
set here.

In [ ]:
import pathlib

KAGGLE = pathlib.Path('/kaggle/input').exists()
COLAB = not KAGGLE and pathlib.Path('/content').exists()

if KAGGLE:
    DATA_ROOT = pathlib.Path('/kaggle/input')   # attached datasets, read only
    SCRATCH = pathlib.Path('/tmp')              # the split lives here: thousands of
                                                # symlinks are not worth saving with the version
    STORE = pathlib.Path('/kaggle/working')     # this is what a saved version keeps
else:
    DATA_ROOT = SCRATCH = STORE = pathlib.Path('/content')
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        STORE = pathlib.Path('/content/drive/MyDrive/bird-classifier')
    except Exception as error:
        print('no Drive - checkpoints and the cached dataset will die with the runtime:', error)

STORE.mkdir(parents=True, exist_ok=True)

# A restart means downloading 16 GB again before anything can resume, so the first run
# leaves a shrunk copy of just the 25 species here (~1-2 GB) and later runs use that.
CACHE_TAR = STORE / 'birds25-256.tar'
CACHE_DIR = SCRATCH / 'birds25-256'
CACHED = CACHE_TAR.exists()

print('running on', 'Kaggle' if KAGGLE else 'Colab' if COLAB else 'something else')
print('data', DATA_ROOT, '| scratch', SCRATCH, '| keeping things in', STORE)
print('cached copy of the dataset:', 'yes' if CACHED else 'not yet')

## 1. Kaggle credentials

Colab only - on Kaggle the dataset is attached to the notebook and this cell does nothing.

Kaggle -> your profile -> Settings -> API -> *Create New Token*. That downloads
`kaggle.json`. Run the cell and upload it - it stays in this Colab session only.

In [ ]:
import os

if KAGGLE:
    print('nothing to do - the dataset is attached to this notebook')
else:
    from google.colab import files
    if not pathlib.Path('/root/.kaggle/kaggle.json').exists():
        print('Upload your kaggle.json:')
        uploaded = files.upload()
        os.makedirs('/root/.kaggle', exist_ok=True)
        with open('/root/.kaggle/kaggle.json', 'wb') as f:
            f.write(next(iter(uploaded.values())))
        os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print('credentials ready')

## 2. Get the dataset

On Kaggle it is already mounted under `/kaggle/input`, read-only, and this is instant.
On Colab it downloads ~16 GB, which takes a few minutes - the disk is temporary, which is
exactly why we train up here instead of on a laptop.

In [ ]:
if CACHED:
    print('there is a cached copy on Drive - skipping the 16 GB download')
elif KAGGLE:
    print('already attached:', [d.name for d in DATA_ROOT.iterdir()])
else:
    !pip install -q kaggle
    !kaggle datasets download -d pavangawande/birds25-cleaned -p /content --unzip
    !du -sh /content/* | head

In [ ]:
import pathlib, tarfile, time

# find the folder that actually holds the per-species subfolders
def find_image_root(start):
    best, best_count = None, 0
    for path in pathlib.Path(start).rglob('*'):
        if not path.is_dir():
            continue
        subdirs = [d for d in path.iterdir() if d.is_dir()]
        if len(subdirs) < 10:
            continue
        images = sum(1 for d in subdirs for _ in d.glob('*.jpg'))
        if images > best_count:
            best, best_count = path, images
    return best, best_count

if CACHED:
    started = time.time()
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    with tarfile.open(CACHE_TAR) as tar:
        try:
            tar.extractall(CACHE_DIR, filter='data')   # python 3.12 wants this spelled out
        except TypeError:
            tar.extractall(CACHE_DIR)
    IMAGE_ROOT = CACHE_DIR
    total = sum(1 for _ in CACHE_DIR.rglob('*.jpg'))
    print(f'unpacked the cached copy in {time.time() - started:.0f}s')
else:
    IMAGE_ROOT, total = find_image_root(DATA_ROOT)

print('images root:', IMAGE_ROOT, '| images:', total)
for d in sorted(IMAGE_ROOT.iterdir())[:30]:
    if d.is_dir():
        print(f'  {d.name:34} {len(list(d.glob("*.jpg")))}')

## 3. Match the app's 25 species

The app can only name the species it has sounds for, so we train on exactly those and
in the same order - the ONNX output index has to line up with the app's class list.

In [ ]:
CLASS_NAMES = [
    'Asian Green Bee Eater', 'Brown Headed Barbet', 'Cattle Egret',
    'Common Kingfisher', 'Common Myna', 'Common Rosefinch',
    'Common Tailorbird', 'Coppersmith Barbet', 'Forest Wagtail',
    'Gray Wagtail', 'Hoopoe', 'House Crow',
    'Indian Grey Hornbill', 'Indian Peacock', 'Indian Pitta',
    'Indian Roller', 'Jungle Babbler', 'Northern Lapwing',
    'Red Wattled Lapwing', 'Ruddy Shelduck', 'Rufous Treepie',
    'Sarus Crane', 'White Breasted Kingfisher',
    'White Breasted Waterhen', 'White Wagtail',
]

def normalise(name):
    return ''.join(ch for ch in name.lower() if ch.isalnum())

folders = {normalise(d.name): d for d in IMAGE_ROOT.iterdir() if d.is_dir()}

matched, missing = {}, []
for species in CLASS_NAMES:
    folder = folders.get(normalise(species))
    if folder is None:  # try a looser match
        candidates = [f for key, f in folders.items() if normalise(species) in key or key in normalise(species)]
        folder = candidates[0] if len(candidates) == 1 else None
    if folder is None:
        missing.append(species)
    else:
        matched[species] = folder

print(f'matched {len(matched)}/{len(CLASS_NAMES)} species')
if missing:
    print('NOT FOUND - check these against the folder list above:')
    for m in missing: print('  ', m)

### 3b. Shrink it once, keep it on Drive

The photos are far bigger than the 224px the model trains on, and decoding them is what
actually holds the GPU up. This makes a 256px copy of just the 25 species we use and tars
it onto Drive - a few minutes now, and every later session skips the 16 GB download and
trains faster. Skipped on Kaggle, where the dataset is already local.

In [ ]:
import os, shutil, tarfile, time
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

SHORT_SIDE = 256

def shrink(job):
    source, target = job
    try:
        with Image.open(source) as image:
            image.draft('RGB', (SHORT_SIDE, SHORT_SIDE))  # decode straight to roughly this size
            image = image.convert('RGB')
            width, height = image.size
            scale = SHORT_SIDE / min(width, height)
            if scale < 1:
                image = image.resize((round(width * scale), round(height * scale)), Image.BILINEAR)
            image.save(target, 'JPEG', quality=88)
        return True
    except Exception:
        return False   # a handful of the files in this dataset are broken

if CACHED or KAGGLE:
    print('nothing to do' + (' - already cached' if CACHED else ' - running on Kaggle'))
else:
    started = time.time()
    jobs = []
    for species, folder in matched.items():
        target_dir = CACHE_DIR / species
        target_dir.mkdir(parents=True, exist_ok=True)
        for source in folder.glob('*'):
            if source.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
                jobs.append((source, target_dir / (source.stem + '.jpg')))

    # threads, not processes: a function defined in a notebook cell can't always be
    # pickled across to a worker process, and PIL lets go of the GIL while it works anyway
    with ThreadPoolExecutor(max_workers=min(8, (os.cpu_count() or 2) * 2)) as pool:
        done = sum(pool.map(shrink, jobs))
    print(f'shrunk {done}/{len(jobs)} images in {time.time() - started:.0f}s')

    # tar locally first: writing a big file straight to Drive is painfully slow
    local_tar = SCRATCH / 'birds25-256.tar'
    with tarfile.open(local_tar, 'w') as tar:
        tar.add(CACHE_DIR, arcname='.')
    shutil.copy(local_tar, CACHE_TAR)
    print(f'cached {CACHE_TAR.stat().st_size / 1e9:.1f} GB to {CACHE_TAR}')

    # train from the smaller copy from here on
    matched = {species: CACHE_DIR / species for species in matched}

## 4. Build train/val splits

A fixed seed and a per-species split, so validation numbers are comparable between runs.

In [ ]:
import random, shutil, os

random.seed(1337)
WORK = SCRATCH / 'split'
VAL_FRACTION = 0.15

if WORK.exists(): shutil.rmtree(WORK)
counts = {}
for species, folder in matched.items():
    images = sorted(p for p in folder.glob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
    random.shuffle(images)
    cut = max(1, int(len(images) * VAL_FRACTION))
    for split, chunk in (('val', images[:cut]), ('train', images[cut:])):
        target = WORK / split / species
        target.mkdir(parents=True, exist_ok=True)
        for src in chunk:
            os.symlink(src, target / src.name)   # symlink: no copying 16 GB around
    counts[species] = (len(images) - cut, cut)

print(f"{'species':34}{'train':>7}{'val':>6}")
for species, (tr, va) in counts.items(): print(f'{species:34}{tr:>7}{va:>6}')
print(f"\ntotal train {sum(t for t,_ in counts.values())}, val {sum(v for _,v in counts.values())}")

## 5. Fine-tune

EfficientNet-B0, pretrained on ImageNet. The backbone already knows edges, feathers and
textures; we mostly teach it the 25 labels. Far better than learning from scratch on a
few thousand photos, which is what the original CNN had to do.

### Somewhere to keep checkpoints

Drive on Colab, the working folder on Kaggle - either way they outlive the runtime, and
the cell below writes one after every epoch.

In [ ]:
import shutil

CKPT_DIR = STORE
if KAGGLE:
    # to carry on from an earlier run, attach that version's output as an input
    # dataset and this picks the checkpoint up
    earlier = sorted(DATA_ROOT.glob('*/checkpoint.pt'))
    if earlier and not (CKPT_DIR / 'checkpoint.pt').exists():
        shutil.copy(earlier[0], CKPT_DIR / 'checkpoint.pt')
        print('carrying on from', earlier[0])

CKPT_DIR.mkdir(parents=True, exist_ok=True)
CKPT = CKPT_DIR / 'checkpoint.pt'          # everything needed to carry on
BEST = CKPT_DIR / 'bird_classifier.pth'    # weights from the best epoch so far
print('checkpoints ->', CKPT_DIR)

In [ ]:
import os, time
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_SIZE, BATCH, EPOCHS = 224, 64, 8
# Run fewer than the full 8 in one sitting: lower this, run the notebook, and run it again
# later - it carries on from the checkpoint. 2 or 3 is a comfortable Colab session.
EPOCHS_THIS_RUN = 8
CUDA = DEVICE == 'cuda'
torch.backends.cudnn.benchmark = True  # every batch is the same shape, let cuDNN tune for it
print('device:', DEVICE)

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
val_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.15)), transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])

train_ds = datasets.ImageFolder(WORK / 'train', train_tf)
val_ds = datasets.ImageFolder(WORK / 'val', val_tf)

# ImageFolder sorts classes alphabetically; make sure that ordering is the one we ship
ORDERED_CLASSES = train_ds.classes
print('classes:', len(ORDERED_CLASSES))

# decoding JPEGs is what actually holds the GPU up, so use every core Colab gave us
WORKERS = min(4, os.cpu_count() or 2)
loader_args = dict(num_workers=WORKERS, pin_memory=CUDA)
if WORKERS:
    loader_args.update(persistent_workers=True, prefetch_factor=4)
train_dl = DataLoader(train_ds, BATCH, shuffle=True, **loader_args)
val_dl = DataLoader(val_ds, BATCH, shuffle=False, **loader_args)

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(ORDERED_CLASSES))
model = model.to(DEVICE, memory_format=torch.channels_last)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, 3e-4, epochs=EPOCHS, steps_per_epoch=len(train_dl))
# half precision on the tensor cores: same accuracy here, roughly half the time per epoch
scaler = torch.amp.GradScaler('cuda', enabled=CUDA)

def batches(x, y):
    x = x.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
    return x, y.to(DEVICE, non_blocking=True)

def evaluate():
    model.eval(); correct = total = 0
    with torch.no_grad(), torch.amp.autocast('cuda', enabled=CUDA):
        for x, y in val_dl:
            x, y = batches(x, y)
            correct += (model(x).argmax(1) == y).sum().item(); total += y.size(0)
    return correct / total

# carry on from wherever the last run stopped
start_epoch, best = 1, 0.0
if CKPT.exists():
    state = torch.load(CKPT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(state['model'])
    optimizer.load_state_dict(state['optimizer'])
    scheduler.load_state_dict(state['scheduler'])
    scaler.load_state_dict(state['scaler'])
    start_epoch, best = state['epoch'] + 1, state['best']
    if start_epoch > EPOCHS:
        print(f'already finished all {EPOCHS} epochs (best {best:.1%}) - delete {CKPT} to start over')
    else:
        print(f'resuming at epoch {start_epoch}, best so far {best:.1%}')

stop_after = min(EPOCHS, start_epoch + EPOCHS_THIS_RUN - 1)
for epoch in range(start_epoch, stop_after + 1):
    model.train(); started = time.time(); running = 0.0
    for x, y in train_dl:
        x, y = batches(x, y)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=CUDA):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        scheduler.step()
        running += loss.item() * x.size(0)

    accuracy = evaluate()
    print(f'epoch {epoch}/{EPOCHS}  loss {running/len(train_ds):.3f}  val acc {accuracy:.1%}  ({time.time()-started:.0f}s)')

    if accuracy > best:
        best = accuracy
        torch.save(model.state_dict(), BEST)
    # written every epoch, not just the good ones - this is what resuming reads
    torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
                'epoch': epoch, 'best': best, 'classes': ORDERED_CLASSES}, CKPT)

print(f'\nbest validation accuracy: {best:.1%}')
if stop_after < EPOCHS:
    print(f'stopped after epoch {stop_after} of {EPOCHS} - rerun the notebook to carry on '
          f'(the cells above are quick once the dataset is cached)')

## 6. Where it still gets confused

Per-species accuracy is more honest than one headline number - it shows which birds the
model actually struggles with (the two kingfishers and the three wagtails, usually).

In [ ]:
from collections import defaultdict

model.load_state_dict(torch.load(BEST, map_location=DEVICE, weights_only=True))
model.eval()

right, seen = defaultdict(int), defaultdict(int)
confused = defaultdict(int)
with torch.no_grad():
    for x, y in val_dl:
        preds = model(x.to(DEVICE)).argmax(1).cpu()
        for true, pred in zip(y.tolist(), preds.tolist()):
            seen[true] += 1
            if true == pred: right[true] += 1
            else: confused[(ORDERED_CLASSES[true], ORDERED_CLASSES[pred])] += 1

print(f"{'species':34}{'accuracy':>10}{'n':>6}")
for i, name in enumerate(ORDERED_CLASSES):
    if seen[i]: print(f'{name:34}{right[i]/seen[i]:>9.0%}{seen[i]:>6}')

print('\nmost common mix-ups:')
for (true, pred), n in sorted(confused.items(), key=lambda kv: -kv[1])[:10]:
    print(f'  {n:>3}x  {true}  ->  {pred}')

## 7. Export for the app

ONNX for the browser, the checkpoint for the desktop version, and a small JSON so the
app knows the class order and preprocessing without guessing.

In [ ]:
# Colab's newer images ship with none of these: onnx for the exporter itself,
# onnxscript for the torch.export path, onnxruntime for the check in the next cell
!pip install -q onnx onnxscript onnxruntime

import json

# export the best epoch, not whatever the last one happened to be
model.load_state_dict(torch.load(BEST, map_location='cpu', weights_only=True))
model.eval().cpu().to(memory_format=torch.contiguous_format)
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)

ONNX_PATH = CKPT_DIR / 'bird_classifier.onnx'
META_PATH = CKPT_DIR / 'classifier_meta.json'

settings = dict(
    input_names=['input'], output_names=['logits'],
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17,
)
try:
    # torch 2.6+ exports through dynamo by default, which wants the onnxscript
    # package installed; the old exporter is fine for a plain CNN and needs nothing
    torch.onnx.export(model, dummy, str(ONNX_PATH), dynamo=False, **settings)
except TypeError:
    torch.onnx.export(model, dummy, str(ONNX_PATH), **settings)   # older torch

meta = {
    'classes': ORDERED_CLASSES,
    'input_size': IMG_SIZE,
    'mean': MEAN,
    'std': STD,
    'detector_input_size': 640,   # the app reads this too - it belongs to the YOLO detector
    'architecture': 'efficientnet_b0',
    'val_accuracy': best,
}
META_PATH.write_text(json.dumps(meta, indent=2))

for file in (ONNX_PATH, BEST, META_PATH):
    print(f'{file.name:24} {file.stat().st_size / 1e6:.1f} MB')
print(json.dumps({k: v for k, v in meta.items() if k != 'classes'}, indent=2))

In [ ]:
# sanity check: ONNX and PyTorch should agree before we ship it
!pip install -q onnxruntime
import onnxruntime as ort, numpy as np

session = ort.InferenceSession(str(ONNX_PATH))
with torch.no_grad():
    torch_out = model(dummy).numpy()
onnx_out = session.run(None, {'input': dummy.numpy()})[0]
print('max difference:', float(np.abs(torch_out - onnx_out).max()))
assert np.allclose(torch_out, onnx_out, atol=1e-4), 'ONNX export does not match PyTorch'
print('ONNX matches PyTorch')

In [ ]:
if KAGGLE:
    print('done - the files are in the Output tab of this notebook version:')
    for file in sorted(STORE.iterdir()):
        print('  ', file.name)
else:
    from google.colab import files  # they are already on Drive, this is just for the laptop
    for file in (ONNX_PATH, META_PATH):
        files.download(str(file))